# DataMate Multi-Stage Code-Generation Agent (Qwen2.5-Coder-7B-Instruct) — Google Colab Pro

Notebook này được cấu hình chuyên biệt cho **Google Colab Pro (GPU A100 / L4 / T4)** để chạy toàn bộ pipeline **DataMate (Zalo AI Challenge 2026)**.

Hỗ trợ **2 chế độ vận hành**:
1. **Chế độ 1 (Chạy End-to-End 100% trên Colab Pro + Google Drive)**: Mount Google Drive cá nhân, đồng bộ thư mục `zalo AI`, kiểm thử `pytest`, nạp `Qwen/Qwen2.5-Coder-7B-Instruct` lên GPU và chạy `run_pipeline.py` + `evaluate.py`.
2. **Chế độ 2 (Colab Pro làm Remote GPU Server cho máy Mac)**: Mở cổng API tương thích OpenAI qua Cloudflare Tunnel để máy Mac tại `/Applications/zalo AI` gọi trực tiếp GPU Colab Pro.

## Bước 1: Chuẩn bị Môi trường, Tải Code từ GitHub & Nạp Data từ Google Drive
- Tải code mới nhất từ GitHub (`https://github.com/25520969-netizen/Datamate.git`) chỉ mất 0.5 giây.
- Nạp dữ liệu từ file `zalo_data.zip` trên Google Drive vào SSD Colab (chỉ nạp 1 lần đầu duy nhất).

In [ ]:
from google.colab import drive
import os, zipfile
from pathlib import Path

# 1. Mount Google Drive
drive.mount("/content/drive")
WORK_DIR = Path("/content/Datamate")

# 2. Cập nhật Code mới nhất từ GitHub
if not WORK_DIR.exists():
    print("🚀 Đang tải code từ GitHub...")
    !git clone https://github.com/25520969-netizen/Datamate.git $WORK_DIR
    %cd $WORK_DIR
else:
    %cd $WORK_DIR
    print("⚡ Đang cập nhật code mới nhất từ GitHub...")
    !git pull

# 3. Nạp Data từ Drive vào SSD Colab (Chỉ giải nén LẦN ĐẦU TIÊN)
DRIVE_ZIP = Path("/content/drive/MyDrive/zalo_data.zip")
if not (WORK_DIR / "train" / "data").exists():
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError("Chưa tìm thấy 'MyDrive/zalo_data.zip'. Vui lòng tải file lên Google Drive trước!")
    print("📦 Lần đầu chạy: Đang giải nén bộ dữ liệu 183 MB vào SSD Colab...")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        zf.extractall(WORK_DIR)
    print("✅ Đã nạp xong toàn bộ dữ liệu Train & Public Test!")
else:
    print("⚡ Dữ liệu đã có sẵn trên SSD Colab, bỏ qua giải nén!")

# 4. Cài đặt thư viện cần thiết
!pip install -q vllm duckdb openpyxl pyarrow pytest
print("
🎉 Môi trường đã sẵn sàng 100%!")


## Bước 2: Kiểm tra Unit Tests (Tùy chọn)
Xác thực 17 bài test của Pipeline (làm sạch subtotal, unaccent search, date resolver, duckdb sandbox).

In [ ]:
!pytest -v


---
## Bước 3: Lựa chọn 1 trong 3 chế độ chạy Benchmark / Nộp bài
Dưới đây là 3 cell hoàn toàn độc lập, mỗi cell tự động quản lý khởi động vLLM và lưu file kết quả về Drive.

### Lựa chọn 1: Benchmark 300 câu mẫu (Train Sample - Thời gian ~4 đến 5 phút)

In [ ]:
# ==============================================================
# CELL 1: CHẠY BENCHMARK 300 CÂU MẪU (TẬP TRAIN)
# ==============================================================
import subprocess, time, os, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = False  # Đổi thành True nếu muốn tự động tắt GPU sau khi chạy xong
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Khởi động vLLM nếu chưa chạy
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 2. Chạy 300 câu benchmark và hiển thị log trực tiếp
print("🚀 [1/2] Bắt đầu benchmark 300 câu mẫu (30 doanh nghiệp)...")
!python3 run_pipeline.py     --questions train/sample_300_questions.jsonl     --gold train/sample_300_answers.jsonl     --data-root train/data     --calendar train/calendar.csv     --rulebook train/rulebook.md     --output predictions_sample300.jsonl     --backend openai     --api-base http://localhost:8000/v1     --concurrency $CONCURRENCY     --evaluate     --fresh

# 3. Lưu kết quả về Google Drive
print("
💾 [2/2] Đang lưu file kết quả về Google Drive...")
local_pred = Path("predictions_sample300.jsonl")
drive_pred = Path("/content/drive/MyDrive/predictions_sample300.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("
🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()


### Lựa chọn 2: Benchmark toàn bộ 3.000 câu Train (Thời gian ~35 đến 45 phút)

In [ ]:
# ==============================================================
# CELL 2: CHẠY BENCHMARK TOÀN BỘ 3.000 CÂU (TẬP TRAIN FULL)
# ==============================================================
import subprocess, time, os, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = True   # Tự động tắt GPU sau khi chạy xong để tiết kiệm Compute Units
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Khởi động vLLM nếu chưa chạy
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 2. Chạy 3.000 câu benchmark và hiển thị log trực tiếp
print("🚀 [1/2] Bắt đầu benchmark toàn bộ 3.000 câu tập Train...")
!python3 run_pipeline.py     --questions train/questions.jsonl     --gold train/answers.jsonl     --data-root train/data     --calendar train/calendar.csv     --rulebook train/rulebook.md     --output predictions_train3000.jsonl     --backend openai     --api-base http://localhost:8000/v1     --concurrency $CONCURRENCY     --evaluate     --fresh

# 3. Lưu kết quả về Google Drive
print("
💾 [2/2] Đang lưu file kết quả về Google Drive...")
local_pred = Path("predictions_train3000.jsonl")
drive_pred = Path("/content/drive/MyDrive/predictions_train3000.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("
🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()


### Lựa chọn 3: Chạy 1.000 câu Public Test nộp thi Zalo AI (Thời gian ~12 đến 15 phút)

In [ ]:
# ==============================================================
# CELL 3: CHẠY 1.000 CÂU THI PUBLIC TEST (XUẤT SUBMISSION.JSONL)
# ==============================================================
import subprocess, time, os, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = True   # Tự động tắt GPU sau khi chạy xong để tiết kiệm Compute Units
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Khởi động vLLM nếu chưa chạy
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 2. Chạy 1.000 câu Public Test (dev_001 -> dev_014)
print("🚀 [1/2] Bắt đầu chạy 1.000 câu Public Test nộp bài thi Zalo AI...")
!python3 run_pipeline.py     --questions public_test/questions.jsonl     --data-root public_test/data     --calendar public_test/calendar.csv     --rulebook public_test/rulebook.md     --output submission.jsonl     --backend openai     --api-base http://localhost:8000/v1     --concurrency $CONCURRENCY     --fresh

# 3. Lưu file submission.jsonl về Google Drive
print("
💾 [2/2] Đang lưu file nộp bài submission.jsonl về Google Drive...")
local_pred = Path("submission.jsonl")
drive_pred = Path("/content/drive/MyDrive/submission.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu file nộp bài tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("
🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()
